In [1]:
!pip install -q spacy altair GPUtil 

In [2]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 88.0 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
from os.path import exists

import sys
import math
import copy
import time
import spacy
import GPUtil

In [4]:
from pathlib import Path
import pandas as pd
import altair as alt

In [ ]:
import urllib.request
import shutil
import tarfile
from collections import Counter
from itertools import chain

In [6]:
import torch

import torch.nn as nn
from torch.nn.functional import log_softmax, pad
from torch.nn.parallel import DistributedDataParallel as DDP # Cf., Lecture 12

from torch.optim.lr_scheduler import LambdaLR

from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

import torch.distributed as dist
import torch.multiprocessing as mp

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
RUN_EXAMPLES = True

def is_interactive_notebook():
    return __name__ == "__name__"

def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)

def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)

In [9]:
class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr":0}]
        None
    def step(self):
        None
    def zero_grad(self, set_to_none = False):
        None

class DummyScheduler:
    def step(self):
        None

# Part 1: Model Architecture

## Encoder-Decoder

In [10]:
class EncoderDecoder(nn.Module):
    """
    A Standard Encoder-Decoder Architecture.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()

        self.encoder = encoder
        self.decoder = decoder

        self.src_embed = src_embed
        self.tgt_embed = tgt_embed

        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)
    
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [ ]:
class Generator(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)